# PRIMA uncertainty-aware articulation diagnostics

This notebook diagnoses the real `fox_02` failure from `output/smal_r_prima_uncertainty` and compares it with the legacy PRIMA result. It reconstructs the three selected, surface-optimized SMAL hypotheses from the saved production parameter files, renders them in their original input cameras, expands confidence-weighted visibility fusion, and checks the reconstructed results against the production OBJ files.

The values below mirror `uncertainty_aware_prima`: 18 production camera views, three surface views, confidence-weighted fusion, unit residual weights, temperature 1, and visibility tolerance $10^{-3}$. The notebook does not rerun PRIMA inference or optimization; it uses the exact selected-view parameters and diagnostics saved by production. Run it with `pkg/PRIMA/.venv/bin/python`.

## 1. Setup and explicit artifacts

In [ ]:
import os
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/atm-matplotlib")

import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "articulate").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ATM repository root")

PROJECT_ROOT = find_project_root()
PRIMA_ROOT = PROJECT_ROOT / "pkg/PRIMA"
for path in (PROJECT_ROOT, PRIMA_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(PROJECT_ROOT)

RAW_PATH = Path("/home/knpob/work/data/shape-corr/SMAL_r/off/fox_02.off")
UNCERTAINTY_ROOT = PROJECT_ROOT / "output/smal_r_prima_uncertainty"
LEGACY_ROOT = PROJECT_ROOT / "output/smal_r_prima"
SMAL_MODEL_PATH = PRIMA_ROOT / "data/smal/my_smpl_00781_4_all.pkl"
DEVICE = "cpu"
FUSION_EPS = 1e-8

required = [
    RAW_PATH, SMAL_MODEL_PATH,
    UNCERTAINTY_ROOT / "smal/fox_02.obj",
    UNCERTAINTY_ROOT / "smal/fox_02.npy",
    UNCERTAINTY_ROOT / "smal/fox_02.confidence.npz",
    LEGACY_ROOT / "smal/fox_02.obj",
    LEGACY_ROOT / "smal/fox_02.npy",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing required artifacts:\n" + "\n".join(missing)
print(f"Project: {PROJECT_ROOT}")
print(f"Input:   {RAW_PATH}")

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation

from articulate.shape_loader.animer_mesh_loader import AniMerMeshShapeLoader
from articulate.shape_prior.prima_smal_prior import PRIMASMALShapePrior
from articulate.shape_refiner.uncertainty_aware_mhr_refiner import confidence_weighted_fusion
from articulate.view_sampler.multi_view_sampler import MultiViewSampler, render_mesh
from prima.models.smal_wrapper import SMAL

pv.set_jupyter_backend("static")
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

def load_mesh(path):
    mesh = trimesh.load(path, process=False)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected a triangular mesh at {path}")
    return mesh

def load_params(root):
    return np.load(root / "smal/fox_02.npy", allow_pickle=True).item()

def tensor(value):
    return torch.as_tensor(value, dtype=torch.float32, device=DEVICE)

def as_mesh(vertices, faces):
    return trimesh.Trimesh(vertices=np.asarray(vertices), faces=np.asarray(faces), process=False)

def rgba_overlay(rgb, rgba):
    rgb = rgb.astype(np.float32) / 255.0
    rgba = rgba.astype(np.float32) / 255.0
    alpha = rgba[..., 3:4]
    rendered = rgba[..., :3] * alpha + rgb * (1.0 - alpha)
    return np.clip(0.55 * rgb + 0.45 * rendered, 0.0, 1.0)

## 2. Load the raw shape, final meshes, selected-view parameters, and confidence sidecar

The animal loader applies the same 180° x-axis rotation used by both production PRIMA presets. The saved parameter arrays contain one optimized SMAL hypothesis and similarity transform for each of the three views retained for surface optimization.

In [ ]:
shape_loader = AniMerMeshShapeLoader(shape_dir=str(RAW_PATH.parent))
shape = shape_loader.pipeline(str(RAW_PATH))
raw_mesh = shape["mesh"]
uncertainty_mesh = load_mesh(UNCERTAINTY_ROOT / "smal/fox_02.obj")
legacy_mesh = load_mesh(LEGACY_ROOT / "smal/fox_02.obj")
uncertainty_params = load_params(UNCERTAINTY_ROOT)
legacy_params = load_params(LEGACY_ROOT)
with np.load(UNCERTAINTY_ROOT / "smal/fox_02.confidence.npz", allow_pickle=False) as archive:
    confidence = {name: np.asarray(archive[name]) for name in archive.files}

view_keys = [tuple(values) for values in confidence["view_keys"]]
view_count = len(view_keys)
vertex_count = len(uncertainty_mesh.vertices)
assert view_count == uncertainty_params["betas"].shape[0] == 3
assert vertex_count == confidence["vertex_uncertainty"].shape[0] == 3889
print(f"Selected views: {view_keys}")
print(f"View weights:   {np.round(confidence['view_weights'], 4).tolist()}")
print(f"Never-visible vertices: {confidence['vertex_visibility_fallback'].mean():.1%}")

In [ ]:
plotter = pv.Plotter(shape=(1, 3), window_size=(1650, 540))
for column, (title, fitted) in enumerate([
    ("Legacy PRIMA mean fusion", legacy_mesh),
    ("Uncertainty-aware fusion", uncertainty_mesh),
    ("Difference between final meshes", uncertainty_mesh),
]):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(raw_mesh), color="lightblue", opacity=0.30, label="input")
    plotter.add_mesh(pv.wrap(fitted), color="salmon", opacity=0.72, label="SMAL")
    if column == 2:
        plotter.add_mesh(pv.wrap(legacy_mesh), color="gold", opacity=0.32, label="legacy")
    plotter.camera_position = "zy"
    plotter.add_title(title)
    plotter.add_legend()
plotter.show()

## 3. Render the exact selected input views

The three selected cameras occupy only a 90° azimuth range. This section uses the production sampler and camera convention, so the RGB/depth images are the observations that originally drove PRIMA.

In [ ]:
view_sampler = MultiViewSampler(
    elevation_config="multi_3level", rotation_step=45.0, camera_dist=4.0, axis_order="xyz",
    camera_configs={"focal_length": 1000.0, "camera_center": (256, 256), "h": 512, "w": 512},
)
view_sampling = view_sampler.pipeline(shape=shape, camera_views=view_sampler.sample_views())
observations = view_sampling["observations"]
selected_observations = [observations[key] for key in view_keys]
fig, axes = plt.subplots(2, view_count, figsize=(4 * view_count, 8), constrained_layout=True)
for index, (key, observation) in enumerate(zip(view_keys, selected_observations)):
    axes[0, index].imshow(observation["rgb"])
    axes[0, index].set_title(f"selected {key}")
    axes[1, index].imshow(observation["depth"], cmap="viridis")
    axes[1, index].set_title("input depth")
    axes[0, index].axis("off"); axes[1, index].axis("off")
plt.show()

## 4. Reconstruct the selected SMAL hypotheses and verify production equivalence

The fixed SMAL layer is loaded from PRIMA's configured model asset; the production `PRIMASMALShapePrior.generate()` method applies pose, the animal coordinate conversion, and each view's saved similarity transform. No predictor or optimizer behavior is reimplemented here.

In [ ]:
with SMAL_MODEL_PATH.open("rb") as model_file:
    smal_data = pickle.load(model_file, encoding="latin1")
smal_model = SMAL(**smal_data).to(DEVICE).eval()
shape_prior = PRIMASMALShapePrior(device=DEVICE)
shape_prior.smal_model = smal_model
shape_prior.faces = smal_model.faces.detach().cpu().numpy()

def reconstruct(params):
    with torch.no_grad():
        vertices, joints = shape_prior.generate(
            tensor(params["betas"]), tensor(params["go_aa"]), tensor(params["pose_aa"]),
            tensor(params["R"]), tensor(params["s"]), tensor(params["t"]),
        )
    return vertices.cpu().numpy(), joints.cpu().numpy()

uncertainty_vertices, uncertainty_joints = reconstruct(uncertainty_params)
legacy_vertices, legacy_joints = reconstruct(legacy_params)
legacy_mean = legacy_vertices.mean(axis=0)
manual_fused, manual_uncertainty, manual_support, manual_fallback = confidence_weighted_fusion(
    uncertainty_vertices, confidence["view_weights"], confidence["vertex_visibility"], eps=FUSION_EPS
)
legacy_error = np.linalg.norm(legacy_mean - legacy_mesh.vertices, axis=1)
fusion_error = np.linalg.norm(manual_fused - uncertainty_mesh.vertices, axis=1)
np.testing.assert_allclose(legacy_mean, legacy_mesh.vertices, atol=2e-6)
np.testing.assert_allclose(manual_fused, uncertainty_mesh.vertices, atol=2e-6)
np.testing.assert_allclose(manual_uncertainty, confidence["vertex_uncertainty"], atol=1e-7)
print(f"Legacy mean reproduction max error: {legacy_error.max():.2e}")
print(f"Confidence fusion reproduction max error: {fusion_error.max():.2e}")

In [ ]:
overlay_images = []
for vertices, observation in zip(uncertainty_vertices, selected_observations):
    fitted = as_mesh(vertices, shape_prior.faces)
    rgba, _ = render_mesh(
        fitted, camera_pose=observation["cam"]["cam_pose"],
        mesh_base_color=(0.85, 0.25, 0.20), **observation["cam"]["intrinsics"]
    )
    overlay_images.append(rgba_overlay(observation["rgb"], rgba))
fig, axes = plt.subplots(1, view_count, figsize=(4 * view_count, 4), constrained_layout=True)
for index, (key, image) in enumerate(zip(view_keys, overlay_images)):
    axes[index].imshow(image); axes[index].set_title(f"fitted hypothesis {key}"); axes[index].axis("off")
plt.show()

In [ ]:
plotter = pv.Plotter(shape=(1, view_count), window_size=(1650, 540))
colors = ["tomato", "gold", "mediumseagreen"]
for index, (key, vertices, weight) in enumerate(zip(view_keys, uncertainty_vertices, confidence["view_weights"])):
    plotter.subplot(0, index)
    plotter.add_mesh(pv.wrap(raw_mesh), color="lightblue", opacity=0.25, label="input")
    plotter.add_mesh(pv.wrap(as_mesh(vertices, shape_prior.faces)), color=colors[index], opacity=0.76, label="view fit")
    plotter.camera_position = "zy"
    plotter.add_title(f"{key}; global weight={weight:.3f}")
    plotter.add_legend()
plotter.show()

## 5. Residuals and global view weights

Production stores alignment for diagnosis but excludes it from the confidence residual. For this example the IoU and inlier penalties are orders of magnitude larger than surface and keypoint terms, while temperature 1 keeps the three weights relatively flat.

In [ ]:
names = confidence["residual_names"].tolist()
terms = confidence["view_residual_terms"].astype(float)
residuals = confidence["view_residuals"].astype(float)
weights = confidence["view_weights"].astype(float)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
image = axes[0].imshow(terms, aspect="auto", cmap="magma")
axes[0].set_xticks(range(len(names)), names, rotation=35, ha="right")
axes[0].set_yticks(range(view_count), [str(key) for key in view_keys])
axes[0].set_title("Stored residual terms"); fig.colorbar(image, ax=axes[0])
axes[1].bar(range(view_count), residuals); axes[1].set_xticks(range(view_count), [str(k) for k in view_keys], rotation=25); axes[1].set_title("Combined residual (lower is better)")
axes[2].bar(range(view_count), weights); axes[2].set_xticks(range(view_count), [str(k) for k in view_keys], rotation=25); axes[2].set_title("Softmax view weight")
plt.show()

## 6. Visibility, fallback, fusion, and uncertainty

Visibility turns each global weight into a per-vertex weight. A fallback vertex was invisible from every selected view, so its fused position is an ordinary weighted average of all three hypotheses. This is a geometric safety fallback, not evidence that the anatomy is correct.

In [ ]:
visibility = confidence["vertex_visibility"].astype(float)
plotter = pv.Plotter(shape=(1, view_count), window_size=(1650, 540))
for index, (key, vertices) in enumerate(zip(view_keys, uncertainty_vertices)):
    plotter.subplot(0, index)
    poly = pv.wrap(as_mesh(vertices, shape_prior.faces))
    poly.point_data["visible"] = visibility[index]
    plotter.add_mesh(poly, scalars="visible", cmap=["dimgray", "limegreen"], clim=(0, 1), show_scalar_bar=index == view_count - 1)
    plotter.camera_position = "zy"
    plotter.add_title(f"{key}: {visibility[index].mean():.1%} visible")
plotter.show()

In [ ]:
np.testing.assert_allclose(manual_support, confidence["vertex_visibility_support"], atol=1e-7)
np.testing.assert_array_equal(manual_fallback, confidence["vertex_visibility_fallback"].astype(bool))
fields = [
    ("weighted visibility support", manual_support, "viridis", (0, 1)),
    ("never-visible fallback", manual_fallback.astype(float), "gray_r", (0, 1)),
    ("log10 cross-view variance", np.log10(manual_uncertainty + 1e-10), "inferno", None),
]
plotter = pv.Plotter(shape=(1, 3), window_size=(1650, 540))
for column, (title, values, cmap, clim) in enumerate(fields):
    plotter.subplot(0, column)
    poly = pv.wrap(as_mesh(manual_fused, shape_prior.faces))
    poly.point_data[title] = values
    plotter.add_mesh(poly, scalars=title, cmap=cmap, clim=clim)
    plotter.add_mesh(pv.wrap(raw_mesh), color="lightblue", opacity=0.12)
    plotter.camera_position = "zy"
    plotter.add_title(title)
plotter.show()
print(f"Union visibility coverage: {(~manual_fallback).mean():.1%}")

## 7. Pose disagreement and surface-distance diagnostics

A global view residual can hide a locally implausible joint. The first plot measures each of the 34 SMAL pose rotations against the weighted mean rotation across selected hypotheses. The second compares nearest-input-surface distances; low Chamfer proximity does not imply a valid articulation because limbs can fold onto another part of the target surface.

In [ ]:
pose = uncertainty_params["pose_aa"]
pose_dispersion = np.zeros(pose.shape[1])
for joint_index in range(pose.shape[1]):
    rotations = Rotation.from_rotvec(pose[:, joint_index])
    mean_rotation = rotations.mean(weights=weights)
    deviations = (mean_rotation.inv() * rotations).magnitude()
    pose_dispersion[joint_index] = np.sqrt(np.sum(weights * deviations**2))

raw_tree = cKDTree(raw_mesh.vertices)
per_view_distances = np.stack([raw_tree.query(vertices)[0] for vertices in uncertainty_vertices])
final_distance = raw_tree.query(manual_fused)[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 4), constrained_layout=True)
axes[0].bar(np.arange(len(pose_dispersion)), np.degrees(pose_dispersion))
axes[0].set(title="Cross-view pose rotation dispersion", xlabel="SMAL pose joint ID", ylabel="weighted RMS degrees")
axes[1].bar(np.arange(view_count + 1), [*per_view_distances.mean(axis=1), final_distance.mean()])
axes[1].set_xticks(np.arange(view_count + 1), [*[str(k) for k in view_keys], "fused"], rotation=25)
axes[1].set(title="Mean nearest-input-vertex distance", ylabel="mesh units")
plt.show()
top = np.argsort(pose_dispersion)[-8:][::-1]
print("Most inconsistent SMAL pose joints:", [(int(i), round(float(np.degrees(pose_dispersion[i])), 2)) for i in top])

In [ ]:
poly = pv.wrap(as_mesh(manual_fused, shape_prior.faces))
poly.point_data["nearest input distance"] = final_distance
plotter = pv.Plotter(shape=(1, 2), window_size=(1200, 540))
for column, camera in enumerate(["xy", "zy"]):
    plotter.subplot(0, column)
    plotter.add_mesh(poly, scalars="nearest input distance", cmap="turbo")
    plotter.add_mesh(pv.wrap(raw_mesh), color="lightblue", opacity=0.18)
    plotter.camera_position = camera
    plotter.add_title(f"Fused-to-input distance ({camera})")
plotter.show()

## 8. Why this can twist, and recommended fixes

The failure occurs before fusion. In the shared SMAL lifecycle, `shape_optim()` selects three views using the pose-refined meshes, then reloads those views' original PRIMA parameters before surface optimization. Both current PRIMA presets set `bone_term_weight=0` and `keypoint_term_weight=0`; consequently the surface stage optimizes per-view betas and pose using bidirectional Chamfer alone. Chamfer permits a leg, tail, or torso to fold onto the wrong target region while still lowering distance. The legacy mean and uncertainty-aware fusion therefore receive already twisted shared-topology hypotheses.

Prioritized fixes:

1. **Preserve the multi-view pose solution.** In a new refiner variant, subset the pose-optimized tensors when selecting views instead of restarting from raw PRIMA estimates. This is the most direct correction.
2. **Keep triangulated joints active during surface fitting.** Start with `bone_term_weight=1.0` and `keypoint_term_weight=0.1` as an ablation, then tune after inspecting loss magnitudes. Add a SMAL pose prior or joint-limit penalty if available.
3. **Select complementary views.** Replace top-three Chamfer selection with a score that also enforces azimuth/elevation diversity and sufficient union visibility; the current selected views span only 90° and leave more than half of this template unseen.
4. **Gate anatomically invalid hypotheses before fusion.** Reject or downweight fits using joint reprojection, bone-length error, pose-prior energy, self-intersection, normal flips, and held-out-view silhouette/depth error.
5. **Make confidence locally discriminative.** The current residual is dominated by global IoU/inlier terms and temperature 1 produces nearly uniform weights. Calibrate term scales, lower the temperature only after adding plausibility diagnostics, and consider body-region weights rather than one scalar per view.
6. **Do not expect averaging to untwist topology.** For disputed regions, select a plausible medoid hypothesis or optimize a consensus SMAL parameter set. Averaging corresponding vertices from several folded poses can create another invalid pose even when uncertainty is correctly high.

## Visual correctness checklist

- The selected-view images must be exactly `(180,30)`, `(225,0)`, and `(270,0)` for the saved `fox_02` sidecar.
- Each rendered red hypothesis should align with its corresponding raw RGB observation; a camera-axis error produces an obvious mirrored or displaced overlay.
- Both production-equivalence assertions must pass below $2\times10^{-6}$.
- Green visibility regions should face the corresponding selected camera; a vertical flip indicates the wrong image-row convention.
- The fallback map should cover about 56% of vertices for this failure. Large coherent fallback regions mean the selected camera set cannot support visibility-aware fusion.
- Bright pose-dispersion or uncertainty regions should be checked against folded limbs, tail, neck, and torso. Low surface distance in a visibly folded region is a Chamfer ambiguity, not evidence of correctness.